# PRDForge: from conversations to a traceable PRD
**Use case (from the use-case sheet, healthcare and pharma analytics):** take messy meeting transcripts, Slack threads
and customer feedback, synthesise them, and draft a structured PRD with user stories and acceptance criteria.

**What you will practise:** evidence numbering for traceability, map-reduce with LangGraph's `Send` API (parallel extraction),
conflict detection across stakeholders, nested Pydantic schemas, and a **deterministic critic + revise loop** with a budget.

Cells marked **LIVE** call OpenAI; everything else runs offline.

## 0. Setup
Runs in Google Colab or locally. In Colab the first cell clones the repo and installs this project's requirements; locally, start Jupyter from the project folder.

In [ ]:
import os, sys, subprocess
from pathlib import Path

PROJECT = "04-use-case-projects/prdforge-transcripts-to-prd"
if "google.colab" in sys.modules:
    if not Path("genai-agentic-ai-training").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tarang16/genai-agentic-ai-training.git"], check=True)
    os.chdir(f"genai-agentic-ai-training/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path(PROJECT).exists():  # started from the repo root
    os.chdir(PROJECT)
sys.path.insert(0, os.getcwd())
print("Working in", os.getcwd())

In [ ]:
# LIVE: only needed for the cells marked LIVE. Never paste a key into a cell; getpass keeps it out of the file.
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## 1. Sources become numbered evidence
Two meetings, a Slack channel export and a feedback CSV about a pharma field-sales feature. Every line gets an id the model must cite. PII (an email in the feedback) is scrubbed on load.

In [ ]:
import pandas as pd
from prdforge import load_sources
from prdforge.config import SOURCES_DIR
sources = load_sources(SOURCES_DIR)
rows = [{"id": it["id"], "source": s["title"], "who": it["speaker"], "text": it["text"]} for s in sources for it in s["items"]]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows).head(12)

Notice the disagreements already visible in the raw data: the Head of Sales wants **daily** alerts, Field Ops and the reps want a **weekly** Monday digest; data lands weekly anyway. A good PRD surfaces this; a bad one silently picks a side.

## 2. The critic is code
These checks are what a product lead does by eye. Try them on a deliberately bad PRD.

In [ ]:
from prdforge import critique
from prdforge.sources import evidence_index
evidence = evidence_index(sources)
bad = {"title": "x", "problem_statement": "x", "non_goals": [], "personas": [], "risks": [], "decisions": [], "open_questions": [],
       "goals": [{"goal": "Reps act sooner", "metric": "faster", "evidence_ids": ["M1-21"]}],
       "constraints": [],
       "user_stories": [{"id": "US-01", "as_a": "rep", "i_want": "alerts", "so_that": "I act", "priority": "Must",
                         "acceptance_criteria": [{"given": "a drop", "when": "I open the app", "then": "it loads fast"}],
                         "evidence_ids": ["M9-99"]}]}
synthesis = {"conflicts": [{"id": "C1", "topic": "Alert frequency", "positions": []}]}
for issue in critique(bad, evidence, synthesis, constraint_ids={"M1-14", "S-05"}):
    print("-", issue)

## 3. The graph: map-reduce + critique loop

In [ ]:
# LIVE
from prdforge import build_graph, render, coverage
from prdforge.config import get_llm, DEFAULT_CONTEXT
app = build_graph(get_llm(), max_revisions=2)
print(app.get_graph().draw_mermaid())

In [ ]:
# LIVE
state = app.invoke({"source_dir": str(SOURCES_DIR), "product_context": DEFAULT_CONTEXT})
print("insights:", len(state["insights"]), "| conflicts:", [c["topic"] for c in state["synthesis"]["conflicts"]])
print("critique issues per pass:", state["issue_history"], "| coverage:", f"{coverage(state['prd'], state['evidence']):.0%}")

In [ ]:
# LIVE
from IPython.display import Markdown
Markdown(render(state))

## 4. Evaluate
`evaluate.py` checks the PRD against what a competent PM would take from these sources (`data/eval/expectations.json`): conflicts surfaced, constraints captured, out-of-scope items, key story topics.

In [ ]:
# LIVE
import subprocess, sys
print(subprocess.run([sys.executable, "evaluate.py"], capture_output=True, text=True).stdout)

## 5. Take it further
1. Point it at your own (anonymised) transcripts: one `Name: text` line per utterance.
2. Add an LLM-as-judge rubric (clarity, completeness) next to the deterministic critic and compare.
3. Export the stories to Jira with the REST API (as an MCP tool, with human approval).
4. Measure how `max_revisions` trades cost against remaining issues.